# Lake, eddies and currents in 3D

Interactive Plotly view with MITgcm bathymetry, actual `lvl0` eddy cells at their recorded depths, horizontal velocity arrows, and the median-temperature isotherm as a 3D surface. Drag to rotate, scroll to zoom, and click legend entries to hide layers.

Install `numpy pandas matplotlib plotly MITgcmutils nbformat` in your notebook environment if needed. Model paths, timestep and byte order come from `../config.json`. The default catalogue uses the same layout as `plot_map_eddies.ipynb`; set `catalogue` explicitly for other layouts.

The isotherm temperature is the horizontal median of `THETA` at the nearest model level to the modal `h1`. The nearest date in `modal_analysis/modes_parameters.csv` supplies `h1`; set `isotherm_h1` to override it. `isotherm_stride` controls horizontal sampling. Its color shows elevation anomaly relative to the median isotherm height: red indicates sinking and blue indicates rising. Pale, translucent gold and mint eddy sheets distinguish rotation direction without obscuring the isotherm.

The first snapshot selection scans the large catalogue in bounded-memory chunks; subsequent runs reuse a cache. Times must match exactly. Coordinates are model x/y, not geographic east/north. U/V are averaged from bounding staggered faces, with dry faces closed. W is not displayed. Eddy sheets preserve detected cell footprints; their translucency is illustrative, and does not encode eddy intensity. Vertical exaggeration changes visual proportions, not depth labels.

Catalogue cells on dry MITgcm cells are clipped to the wet mask; the count appears in the title and a warning.


Display settings independently enable velocity magnitude slices, arrows, eddies, and the thermocline proxy. Disabled layers are not rendered; disabling eddies skips catalogue access, and disabling the thermocline skips its calculation and modal h1 lookup. Click legend entries to toggle enabled layers interactively; both eddy rotations toggle together. The figure uses the available width and defaults to 1,000 pixels tall. Velocity slices use `arrow_depths`; their color shows horizontal speed in m/s.


Eddy layers have side walls spanning their MITgcm vertical cell interfaces, clipped at the lakebed, and stronger opacity so they remain visible edge-on. These walls represent detected grid layers, not an interpolated eddy boundary between depths.


The upper panel shows total lake kinetic energy (MJ) from `outputs_swirl/ke_eddy/ke_lake.csv` across `date_window_days`. Its red dot follows the displayed snapshot during slider changes and playback, using an exact timestamp match. The curve retains all CSV samples in the window. With a zero-day window, only the selected snapshot is shown. The panel adds 230 pixels above the 3D view.


The energy panel also shows eddy KE in orange on the right axis (MJ), from `ke_eddy.csv` or the existing `ke_eddies.csv`. The blue lake KE curve uses the left axis; the red date marker remains on that curve.


In [1]:
from pathlib import Path
import importlib
import sys

import numpy as np

if not hasattr(np, "NaN"):
    setattr(np, "NaN", np.nan)
if not hasattr(np, "Inf"):
    setattr(np, "Inf", np.inf)

# Support running with the repository root or eddies directory as the working directory.
work = Path.cwd()
if (work / "plot_3d_eddies.py").exists():
    eddies_dir = work
elif (work / "eddies/plot_3d_eddies.py").exists():
    eddies_dir = work / "eddies"
else:
    eddies_dir = work / "python/horizontal_structure_analysis/eddies"
sys.path.insert(0, str(eddies_dir.resolve()))
# Use the existing MITgcm checkout when MITgcmutils is not installed.
try:
    import MITgcmutils
except ImportError:
    local_utils = eddies_dir.resolve().parents[1] / "MITgcm-checkpoint67z/utils/python/MITgcmutils"
    if not local_utils.exists():
        raise ImportError("Install MITgcmutils in this notebook environment")
    sys.path.insert(0, str(local_utils))
if "plot_3d_eddies" in sys.modules:
    plot_module = importlib.reload(sys.modules["plot_3d_eddies"])
else:
    import plot_3d_eddies as plot_module
build_figure = plot_module.build_figure

In [2]:
model = "zug_2025"
timestamp = "2025-07-02 20:30:00"
max_depth = 80.0
arrow_depths = (2, 10, 30, 60)  # metres below surface; nearest model levels
stride = 4                     # one arrow every 4 horizontal cells
exaggeration = 35              # vertical exaggeration
arrow_seconds = 2500          # arrow length = speed × this duration
catalogue = None              # optional explicit path to lvl0.csv
isotherm_h1 = None            # positive depth in metres; defaults to nearest modal date
isotherm_stride = 2           # sample every 2 horizontal cells for the 3D surface
date_window_days = 3         # cursor range centered on timestamp
date_step_hours = 1           # one date choice every 6 hours
output_dir = eddies_dir / "results_3d"
# Display layers: set each independently to True or False.
show_velocity = True        # colored horizontal speed slices at arrow_depths
show_arrows = True          # current direction and speed arrows
show_eddies = True          # catalogue eddy layers
show_thermocline = True     # existing median-temperature isotherm proxy
figure_height = 700        # pixels
figure_width = None         # None fills available width; or set e.g. 1600


In [ ]:
fig = build_figure(timestamp=timestamp, model=model, catalogue=catalogue,
                   max_depth=max_depth, arrow_depths=arrow_depths, stride=stride,
                   exaggeration=exaggeration, arrow_seconds=arrow_seconds,
                   output_dir=output_dir, isotherm_h1=isotherm_h1,
                   isotherm_stride=isotherm_stride,
                   date_window_days=date_window_days,
                   date_step_hours=date_step_hours,
                   show_velocity=show_velocity, show_arrows=show_arrows,
                   show_eddies=show_eddies, show_thermocline=show_thermocline,
                   height=figure_height, width=figure_width)
fig.show(config={"responsive": True})

/tmp/ipykernel_3919567/374228830.py:1: UserWarning: Clipped 1435 dry catalogue cells to the MITgcm wet mask
  fig = build_figure(timestamp=timestamp, model=model, catalogue=catalogue,
/home/leroquan@eawag.wroot.emp-eaw.ch/work_space/python/horizontal_structure_analysis/eddies/plot_3d_eddies.py:498: UserWarning: Clipped 916 dry catalogue cells to the MITgcm wet mask
  frame_fig = build_figure(
/home/leroquan@eawag.wroot.emp-eaw.ch/work_space/python/horizontal_structure_analysis/eddies/plot_3d_eddies.py:498: UserWarning: Clipped 1550 dry catalogue cells to the MITgcm wet mask
  frame_fig = build_figure(
/home/leroquan@eawag.wroot.emp-eaw.ch/work_space/python/horizontal_structure_analysis/eddies/plot_3d_eddies.py:498: UserWarning: Clipped 1469 dry catalogue cells to the MITgcm wet mask
  frame_fig = build_figure(
/home/leroquan@eawag.wroot.emp-eaw.ch/work_space/python/horizontal_structure_analysis/eddies/plot_3d_eddies.py:498: UserWarning: Clipped 1171 dry catalogue cells to the MITgcm we

/home/leroquan@eawag.wroot.emp-eaw.ch/work_space/python/horizontal_structure_analysis/eddies/plot_3d_eddies.py:498: UserWarning: Clipped 1539 dry catalogue cells to the MITgcm wet mask
  frame_fig = build_figure(
/home/leroquan@eawag.wroot.emp-eaw.ch/work_space/python/horizontal_structure_analysis/eddies/plot_3d_eddies.py:498: UserWarning: Clipped 1072 dry catalogue cells to the MITgcm wet mask
  frame_fig = build_figure(
/home/leroquan@eawag.wroot.emp-eaw.ch/work_space/python/horizontal_structure_analysis/eddies/plot_3d_eddies.py:498: UserWarning: Clipped 1267 dry catalogue cells to the MITgcm wet mask
  frame_fig = build_figure(
/home/leroquan@eawag.wroot.emp-eaw.ch/work_space/python/horizontal_structure_analysis/eddies/plot_3d_eddies.py:498: UserWarning: Clipped 1704 dry catalogue cells to the MITgcm wet mask
  frame_fig = build_figure(
/home/leroquan@eawag.wroot.emp-eaw.ch/work_space/python/horizontal_structure_analysis/eddies/plot_3d_eddies.py:498: UserWarning: Clipped 1819 dry ca

output_dir.mkdir(parents=True, exist_ok=True)
output = output_dir / f"{model}_{timestamp.replace(':', '').replace(' ', '_')}_3d.html"
fig.write_html(output, include_plotlyjs=True)
print(output.resolve())